## Yerel kaynak hazırlığı\n\nBu arşivlenmiş proje paketinde dış kaynak bağlantısı kullanılmaz. Gerekli Deney 1 dosyaları repo içindeki yerel kaynak çıktıları, manifestler ve açıklama dosyalarıyla temsil edilir.

In [ ]:
# ============================================================
# 1. FIND MOUTH METADATA FILE
# This cell only searches source package; it does not modify any files.
# ============================================================

from pathlib import Path

search_root = Path("/content/Deney1_Source")

metadata_files = list(search_root.rglob("metadata.csv"))

print(f"Number of metadata.csv files found: {len(metadata_files)}\n")

for index, path in enumerate(metadata_files, start=1):
    print(f"{index}. {path}")

In [ ]:
# ============================================================
# 2. FIND ALL CSV FILES RELATED TO MOUTH DATA
# This cell only searches source package; it does not modify files.
# ============================================================

from pathlib import Path

search_root = Path("/content/Deney1_Source")

all_csv_files = list(search_root.rglob("*.csv"))

print(f"Total CSV files found: {len(all_csv_files)}\n")

mouth_keywords = [
    "mouth",
    "agiz",
    "ağız",
    "dilara",
    "roi",
    "metadata",
    "manifest",
]

related_csv_files = [
    path
    for path in all_csv_files
    if any(
        keyword in str(path).lower()
        for keyword in mouth_keywords
    )
]

print(f"Potentially relevant CSV files: {len(related_csv_files)}\n")

for index, path in enumerate(related_csv_files, start=1):
    print(f"{index}. {path}")

if not related_csv_files:
    print("\nNo related CSV file was found.")
    print("First 50 CSV files in source package:\n")

    for index, path in enumerate(all_csv_files[:50], start=1):
        print(f"{index}. {path}")

In [ ]:
# ============================================================
# 2. CHECK MOUTH ROI OUTPUT DIRECTORY
# This cell only reads the directory; it does not modify files.
# ============================================================

from pathlib import Path

MOUTH_ROI_ROOT = Path(
    "/content/Deney1_Source/AISC Çalışmalar/"
    "Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output"
)

if not MOUTH_ROI_ROOT.exists():
    raise FileNotFoundError(
        f"Mouth ROI directory was not found:\n{MOUTH_ROI_ROOT}"
    )

print("Mouth ROI directory found:")
print(MOUTH_ROI_ROOT)

print("\nFiles and folders inside:\n")

items = sorted(MOUTH_ROI_ROOT.iterdir())

for item in items:
    item_type = "DIR " if item.is_dir() else "FILE"
    print(f"[{item_type}] {item.name}")

csv_files = sorted(MOUTH_ROI_ROOT.rglob("*.csv"))

print(f"\nNumber of CSV files found: {len(csv_files)}")

for index, csv_path in enumerate(csv_files, start=1):
    print(f"{index}. {csv_path}")

In [ ]:
# ============================================================
# 3. INSPECT MOUTH METADATA
# This cell only reads metadata; it does not modify files.
# ============================================================

import pandas as pd
from pathlib import Path

MOUTH_ROI_ROOT = Path(
    "/content/Deney1_Source/AISC Çalışmalar/"
    "Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output"
)

METADATA_PATH = MOUTH_ROI_ROOT / "metadata.csv"

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file was not found:\n{METADATA_PATH}"
    )

metadata_preview = pd.read_csv(METADATA_PATH)

print("Metadata file found:")
print(METADATA_PATH)

print("\nNumber of rows:", len(metadata_preview))
print("\nColumn names:")
print(metadata_preview.columns.tolist())

print("\nFirst 5 rows:")
display(metadata_preview.head())

for column in ["label", "split", "status"]:
    if column in metadata_preview.columns:
        print(f"\n{column.upper()} distribution:")
        print(metadata_preview[column].value_counts(dropna=False))

In [ ]:
# ============================================================
# 4. MOUTH METADATA SCHEMA AND ID QUALITY CHECK
# This cell only audits metadata; it does not modify files.
# ============================================================

print("ALL COLUMN NAMES:\n")

for index, column in enumerate(metadata_preview.columns, start=1):
    print(f"{index:02d}. {column}")

successful_metadata = metadata_preview[
    metadata_preview["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("SUCCESS")
].copy()

print("\nSUCCESSFUL ROWS:", len(successful_metadata))

print(
    "\nDUPLICATED SAMPLE IDs:",
    successful_metadata["sample_id"].duplicated().sum()
)

print(
    "\nMISSING SAMPLE IDs:",
    successful_metadata["sample_id"].isna().sum()
)

print("\nFIRST SUCCESSFUL ROW — ALL FIELDS:\n")
display(successful_metadata.iloc[[0]].T)

possible_path_columns = [
    column
    for column in successful_metadata.columns
    if "path" in column.lower()
]

print("\nPOSSIBLE IMAGE PATH COLUMNS:")
print(possible_path_columns)

print("\nSUCCESSFUL SAMPLE DISTRIBUTION:\n")
print(
    successful_metadata.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

In [ ]:
# ============================================================
# 5. CONFIGURATION
# Edit only this block before the first full run.
# ============================================================

CONFIG = {
    # Mouth ROI metadata
    "metadata_csv": str(METADATA_PATH),

    # Mouth ROI root directory
    "roi_root": str(MOUTH_ROI_ROOT),

    # Experiment outputs
    "output_root": (
        "/content/Deney1_Source/AISC Çalışmalar/"
        "Deneyler/Dilara/Deney 1/Sonuçlar/"
        "SwinV2_TextureFusion_Mouth"
    ),

    # Metadata columns
    "path_column": "mouth_path",
    "label_column": "label",
    "split_column": "split",
    "video_column": "video_id",
    "sample_id_column": "sample_id",
    "status_column": "status",

    # Only successfully extracted mouth ROIs
    "success_status_values": ["success"],

    # Image and model settings
    "image_size": 224,
    "model_name": "swin_v2_t",
    "pretrained": True,
    "freeze_backbone_epochs": 5,
    "unfreeze_last_stages": True,

    # Handcrafted texture settings
    "lbp_radii": [1, 2, 3],
    "lbp_points": [8, 16, 24],

    "glcm_distances": [1, 2, 4],
    "glcm_angles_deg": [0, 45, 90, 135],
    "glcm_levels": 32,

    "gabor_orientations_deg": [0, 30, 60, 90, 120, 150],
    "gabor_frequencies": [0.10, 0.20, 0.30],

    "wavelet": "db2",
    "wavelet_level": 2,

    # Training settings — same as the eye experiment
    "seed": 42,
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 30,
    "learning_rate": 2e-4,
    "backbone_learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "dropout": 0.30,
    "hidden_dim": 256,
    "patience": 8,
    "threshold": 0.50,

    # Validation threshold search
    "threshold_search_min": 0.10,
    "threshold_search_max": 0.90,
    "threshold_search_steps": 161,

    # Training stability
    "amp": True,
    "gradient_clip_norm": 1.0,

    # Development controls
    "smoke_test": True,
    "smoke_train_batches": 2,
    "smoke_val_batches": 2,
    "feature_cache_flush_every": 100,
    "overwrite_feature_cache": False,

    # Only the same full-fusion experiment as the eye study
    "active_mode": "full",
}

assert CONFIG["active_mode"] == "full"
assert Path(CONFIG["metadata_csv"]).exists()
assert Path(CONFIG["roi_root"]).exists()

print("Configuration loaded successfully.")
print("Active mode:", CONFIG["active_mode"])
print("Metadata:", CONFIG["metadata_csv"])
print("Output directory:", CONFIG["output_root"])

In [ ]:
# ============================================================
# 6. IMPORTS, SEED, DEVICE, RUN ID AND OUTPUT DIRECTORIES
# ============================================================

import os
import gc
import io
import math
import time
import copy
import json
import yaml
import random
import hashlib
import warnings

from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import cv2
import joblib
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from skimage.feature import (
    local_binary_pattern,
    graycomatrix,
    graycoprops,
)
from skimage.filters import gabor

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision.models import (
    swin_v2_t,
    Swin_V2_T_Weights,
)
from torchvision import transforms


warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(CONFIG["seed"])


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RUN_ID = datetime.now().strftime(
    f"%Y%m%d_%H%M_mouth_swinv2_texturefusion_seed"
    f"{CONFIG['seed']}"
)

RUN_DIR = Path(CONFIG["output_root"]) / RUN_ID


DIRS = {
    "run": RUN_DIR,
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "features": RUN_DIR / "features",
    "audit": RUN_DIR / "audit",
}


for directory in DIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


config_file = RUN_DIR / "config_resolved.yaml"

with open(
    config_file,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        CONFIG,
        file,
        allow_unicode=True,
        sort_keys=False,
    )


print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: GPU is not active. "
        "Select a T4 GPU before full training."
    )

print("Run ID:", RUN_ID)
print("Run directory:", RUN_DIR)
print("Configuration saved:", config_file)

In [ ]:
# ============================================================
# 7. ATOMIC FILE WRITING HELPERS
# Protect outputs against interrupted or incomplete writes.
# ============================================================

def atomic_write_bytes(
    data: bytes,
    target: Path,
) -> None:
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = target.with_suffix(
        target.suffix + ".tmp"
    )

    with open(temporary, "wb") as file:
        file.write(data)
        file.flush()
        os.fsync(file.fileno())

    os.replace(temporary, target)


def atomic_write_text(
    text: str,
    target: Path,
) -> None:
    atomic_write_bytes(
        text.encode("utf-8"),
        target,
    )


def atomic_write_json(
    payload: Dict[str, Any],
    target: Path,
) -> None:
    content = json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

    atomic_write_text(
        content,
        target,
    )


def atomic_write_csv(
    dataframe: pd.DataFrame,
    target: Path,
) -> None:
    buffer = io.StringIO()

    dataframe.to_csv(
        buffer,
        index=False,
    )

    atomic_write_text(
        buffer.getvalue(),
        target,
    )


def atomic_joblib_dump(
    object_to_save: Any,
    target: Path,
) -> None:
    target = Path(target)

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = target.with_suffix(
        target.suffix + ".tmp"
    )

    joblib.dump(
        object_to_save,
        temporary,
    )

    # Verify that the temporary file is readable.
    _ = joblib.load(temporary)

    os.replace(
        temporary,
        target,
    )


def atomic_torch_save(
    state: Dict[str, Any],
    target: Path,
) -> None:
    target = Path(target)

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = target.with_suffix(
        target.suffix + ".tmp"
    )

    torch.save(
        state,
        temporary,
    )

    loaded = torch.load(
        temporary,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
    }

    if not required_keys.issubset(loaded):
        raise RuntimeError(
            f"Checkpoint validation failed: {temporary}"
        )

    os.replace(
        temporary,
        target,
    )


print("Atomic file writing helpers are ready.")

In [ ]:
# ============================================================
# 8. METADATA LOAD, NORMALIZATION AND QUALITY GATES
# ============================================================

metadata_path = Path(CONFIG["metadata_csv"])

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata file was not found: {metadata_path}"
    )

metadata = pd.read_csv(metadata_path)

print("Total metadata rows:", len(metadata))
print("Number of columns:", len(metadata.columns))


# ------------------------------------------------------------
# Normalize text columns
# ------------------------------------------------------------

metadata[CONFIG["label_column"]] = (
    metadata[CONFIG["label_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

metadata[CONFIG["split_column"]] = (
    metadata[CONFIG["split_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(
        {
            "training": "train",
            "validation": "val",
            "valid": "val",
            "testing": "test",
        }
    )
)

metadata[CONFIG["status_column"]] = (
    metadata[CONFIG["status_column"]]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# Accounting equality
# ------------------------------------------------------------

success_count = (
    metadata[CONFIG["status_column"]] == "success"
).sum()

skipped_count = (
    metadata[CONFIG["status_column"]] == "skipped"
).sum()

error_count = (
    metadata[CONFIG["status_column"]] == "error"
).sum()

other_status_count = (
    len(metadata)
    - success_count
    - skipped_count
    - error_count
)

assert other_status_count == 0, (
    "Unexpected or missing status values were detected."
)

assert len(metadata) == (
    success_count
    + skipped_count
    + error_count
), "Metadata accounting equality failed."


# ------------------------------------------------------------
# Select valid successful mouth ROI records
# ------------------------------------------------------------

valid = metadata[
    metadata[CONFIG["status_column"]].isin(
        CONFIG["success_status_values"]
    )
    & metadata[CONFIG["label_column"]].isin(
        ["real", "fake"]
    )
    & metadata[CONFIG["split_column"]].isin(
        ["train", "val", "test"]
    )
    & metadata[CONFIG["path_column"]].notna()
].copy()


valid["target"] = (
    valid[CONFIG["label_column"]]
    .map(
        {
            "real": 0,
            "fake": 1,
        }
    )
    .astype(int)
)


# ------------------------------------------------------------
# Resolve absolute or relative mouth image paths
# ------------------------------------------------------------

roi_root = Path(CONFIG["roi_root"])


def resolve_mouth_path(path_value: str) -> str:
    image_path = Path(str(path_value))

    if image_path.is_absolute():
        return str(image_path)

    return str(roi_root / image_path)


valid["resolved_image_path"] = (
    valid[CONFIG["path_column"]]
    .astype(str)
    .map(resolve_mouth_path)
)


# ------------------------------------------------------------
# Check whether mouth ROI files actually exist
# ------------------------------------------------------------

valid["file_exists"] = (
    valid["resolved_image_path"]
    .map(lambda path: Path(path).exists())
)

missing_files = valid[
    ~valid["file_exists"]
].copy()

if not missing_files.empty:
    atomic_write_csv(
        missing_files,
        DIRS["audit"] / "missing_mouth_roi_files.csv",
    )

valid = valid[
    valid["file_exists"]
].copy()


if valid.empty:
    raise RuntimeError(
        "No valid mouth ROI image was found."
    )


# ------------------------------------------------------------
# Required uniqueness checks
# ------------------------------------------------------------

duplicate_sample_count = (
    valid[CONFIG["sample_id_column"]]
    .duplicated()
    .sum()
)

assert duplicate_sample_count == 0, (
    f"{duplicate_sample_count} duplicated sample IDs detected."
)


# ------------------------------------------------------------
# Split leakage check using video IDs
# ------------------------------------------------------------

video_sets = {
    split: set(
        valid.loc[
            valid[CONFIG["split_column"]] == split,
            CONFIG["video_column"],
        ].astype(str)
    )
    for split in ["train", "val", "test"]
}

assert video_sets["train"].isdisjoint(
    video_sets["val"]
), "Train/validation video leakage detected."

assert video_sets["train"].isdisjoint(
    video_sets["test"]
), "Train/test video leakage detected."

assert video_sets["val"].isdisjoint(
    video_sets["test"]
), "Validation/test video leakage detected."


# ------------------------------------------------------------
# Save the exact manifest used by the model
# ------------------------------------------------------------

valid = valid.reset_index(drop=True)

atomic_write_csv(
    valid,
    DIRS["audit"] / "valid_mouth_manifest.csv",
)


print("\nMetadata quality gates passed.")
print("Successful metadata rows:", success_count)
print("Skipped metadata rows:", skipped_count)
print("Error metadata rows:", error_count)
print("Missing mouth ROI files:", len(missing_files))
print("Final usable mouth ROI samples:", len(valid))

print("\nUsable sample distribution:")
print(
    valid.groupby(
        [
            CONFIG["split_column"],
            CONFIG["label_column"],
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)

In [ ]:
# ============================================================
# 9. DATA DISTRIBUTION FIGURE
# Minimum short edge: 600 pixels
# Plot language: English
# ============================================================

def save_figure(
    figure: plt.Figure,
    filename: str,
) -> Path:
    target = DIRS["figures"] / filename

    figure.savefig(
        target,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(figure)

    with Image.open(target) as saved_image:
        image_size = saved_image.size

        if min(image_size) < 600:
            raise AssertionError(
                f"Figure resolution is below 600 pixels: "
                f"{image_size}"
            )

    return target


split_class_distribution = (
    valid.groupby(
        [
            CONFIG["split_column"],
            CONFIG["label_column"],
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)


figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

split_class_distribution.plot(
    kind="bar",
    ax=axis,
    color={
        "fake": "#3182CE",
        "real": "#DD6B20",
    },
)

axis.set_title(
    "Mouth ROI Class Distribution by Dataset Split",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Dataset Split",
    fontsize=11,
)

axis.set_ylabel(
    "Number of Samples",
    fontsize=11,
)

axis.legend(
    title="Class",
    frameon=True,
)

axis.grid(
    axis="y",
    alpha=0.25,
)

axis.tick_params(
    axis="x",
    rotation=0,
)

figure.tight_layout()

distribution_figure_path = save_figure(
    figure,
    "01_mouth_roi_dataset_distribution.png",
)

print("Dataset distribution figure saved:")
print(distribution_figure_path)

with Image.open(distribution_figure_path) as saved_image:
    print("Figure resolution:", saved_image.size)

In [ ]:
# ============================================================
# 10. HANDCRAFTED TEXTURE FEATURE EXTRACTION
# LBP + GLCM + Gabor + Wavelet
# ============================================================

def read_gray_image(
    path: str,
    size: int = 224,
) -> np.ndarray:
    image = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise ValueError(
            f"Image could not be read: {path}"
        )

    if image.shape != (size, size):
        image = cv2.resize(
            image,
            (size, size),
            interpolation=cv2.INTER_AREA,
        )

    return image


def safe_entropy(
    values: np.ndarray,
    bins: int = 64,
) -> float:
    flattened = np.asarray(
        values,
        dtype=np.float64,
    ).ravel()

    if flattened.size == 0:
        return 0.0

    histogram, _ = np.histogram(
        flattened,
        bins=bins,
        density=False,
    )

    probabilities = histogram.astype(
        np.float64
    )

    total = probabilities.sum()

    if total <= 0:
        return 0.0

    probabilities /= total
    probabilities = probabilities[
        probabilities > 0
    ]

    return float(
        -(
            probabilities
            * np.log2(probabilities)
        ).sum()
    )


# ------------------------------------------------------------
# LBP FEATURES
# ------------------------------------------------------------

def extract_lbp(
    gray: np.ndarray,
) -> Tuple[np.ndarray, List[str]]:
    features = []
    names = []

    for radius, points in zip(
        CONFIG["lbp_radii"],
        CONFIG["lbp_points"],
    ):
        lbp = local_binary_pattern(
            gray,
            P=points,
            R=radius,
            method="uniform",
        )

        number_of_bins = points + 2

        histogram, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(
                0,
                number_of_bins + 1,
            ),
            range=(
                0,
                number_of_bins,
            ),
        )

        histogram = histogram.astype(
            np.float32
        )

        histogram /= (
            histogram.sum() + 1e-8
        )

        features.extend(
            histogram.tolist()
        )

        names.extend(
            [
                f"lbp_r{radius}_p{points}_bin{index}"
                for index in range(number_of_bins)
            ]
        )

    return (
        np.asarray(
            features,
            dtype=np.float32,
        ),
        names,
    )


# ------------------------------------------------------------
# GLCM FEATURES
# ------------------------------------------------------------

def extract_glcm(
    gray: np.ndarray,
) -> Tuple[np.ndarray, List[str]]:
    quantized = np.floor(
        gray.astype(np.float32)
        / 256.0
        * CONFIG["glcm_levels"]
    ).astype(np.uint8)

    quantized = np.clip(
        quantized,
        0,
        CONFIG["glcm_levels"] - 1,
    )

    angles_radian = [
        np.deg2rad(angle)
        for angle in CONFIG["glcm_angles_deg"]
    ]

    matrix = graycomatrix(
        quantized,
        distances=CONFIG["glcm_distances"],
        angles=angles_radian,
        levels=CONFIG["glcm_levels"],
        symmetric=True,
        normed=True,
    )

    properties = [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM",
    ]

    features = []
    names = []

    for property_name in properties:
        values = graycoprops(
            matrix,
            property_name,
        )

        for distance_index, distance in enumerate(
            CONFIG["glcm_distances"]
        ):
            for angle_index, angle in enumerate(
                CONFIG["glcm_angles_deg"]
            ):
                features.append(
                    float(
                        values[
                            distance_index,
                            angle_index,
                        ]
                    )
                )

                names.append(
                    f"glcm_{property_name}"
                    f"_d{distance}_a{angle}"
                )

    return (
        np.asarray(
            features,
            dtype=np.float32,
        ),
        names,
    )


# ------------------------------------------------------------
# GABOR FEATURES
# ------------------------------------------------------------

def extract_gabor(
    gray: np.ndarray,
) -> Tuple[np.ndarray, List[str]]:
    image = (
        gray.astype(np.float32)
        / 255.0
    )

    features = []
    names = []

    for frequency in CONFIG[
        "gabor_frequencies"
    ]:
        for angle_degree in CONFIG[
            "gabor_orientations_deg"
        ]:
            theta = np.deg2rad(
                angle_degree
            )

            real_response, imaginary_response = gabor(
                image,
                frequency=frequency,
                theta=theta,
            )

            magnitude = np.sqrt(
                real_response**2
                + imaginary_response**2
            )

            statistics = {
                "mean": float(
                    magnitude.mean()
                ),
                "std": float(
                    magnitude.std()
                ),
                "energy": float(
                    np.mean(
                        magnitude**2
                    )
                ),
                "entropy": safe_entropy(
                    magnitude
                ),
            }

            for statistic_name, value in (
                statistics.items()
            ):
                features.append(value)

                names.append(
                    f"gabor_f{frequency:.2f}"
                    f"_a{angle_degree}"
                    f"_{statistic_name}"
                )

    return (
        np.asarray(
            features,
            dtype=np.float32,
        ),
        names,
    )


# ------------------------------------------------------------
# WAVELET FEATURES
# ------------------------------------------------------------

def coefficient_statistics(
    coefficients: np.ndarray,
    prefix: str,
) -> Tuple[List[float], List[str]]:
    array = np.asarray(
        coefficients,
        dtype=np.float64,
    )

    absolute_array = np.abs(array)

    statistics = {
        "mean": float(array.mean()),
        "std": float(array.std()),
        "energy": float(
            np.mean(array**2)
        ),
        "abs_mean": float(
            absolute_array.mean()
        ),
        "abs_max": float(
            absolute_array.max()
        ),
        "entropy": safe_entropy(
            array
        ),
    }

    values = list(
        statistics.values()
    )

    names = [
        f"{prefix}_{name}"
        for name in statistics
    ]

    return values, names


def extract_wavelet(
    gray: np.ndarray,
) -> Tuple[np.ndarray, List[str]]:
    image = (
        gray.astype(np.float32)
        / 255.0
    )

    coefficients = pywt.wavedec2(
        image,
        wavelet=CONFIG["wavelet"],
        level=CONFIG["wavelet_level"],
        mode="symmetric",
    )

    features = []
    names = []

    approximation = coefficients[0]

    values, value_names = (
        coefficient_statistics(
            approximation,
            (
                f"wavelet_L"
                f"{CONFIG['wavelet_level']}_LL"
            ),
        )
    )

    features.extend(values)
    names.extend(value_names)

    detail_levels = coefficients[1:]
    current_level = CONFIG[
        "wavelet_level"
    ]

    for (
        horizontal,
        vertical,
        diagonal,
    ) in detail_levels:
        for band_name, band in [
            ("LH", horizontal),
            ("HL", vertical),
            ("HH", diagonal),
        ]:
            values, value_names = (
                coefficient_statistics(
                    band,
                    (
                        f"wavelet_L"
                        f"{current_level}_"
                        f"{band_name}"
                    ),
                )
            )

            features.extend(values)
            names.extend(value_names)

        current_level -= 1

    return (
        np.asarray(
            features,
            dtype=np.float32,
        ),
        names,
    )


# ------------------------------------------------------------
# COMBINE ALL TEXTURE FEATURES
# ------------------------------------------------------------

def extract_texture_features(
    path: str,
) -> Tuple[
    np.ndarray,
    Dict[str, slice],
    List[str],
]:
    gray = read_gray_image(
        path,
        CONFIG["image_size"],
    )

    feature_groups = {}
    all_features = []
    all_names = []

    start_index = 0

    extractors = [
        ("lbp", extract_lbp),
        ("glcm", extract_glcm),
        ("gabor", extract_gabor),
        ("wavelet", extract_wavelet),
    ]

    for group_name, extractor in extractors:
        values, names = extractor(gray)

        end_index = (
            start_index + len(values)
        )

        feature_groups[group_name] = slice(
            start_index,
            end_index,
        )

        start_index = end_index

        all_features.append(values)
        all_names.extend(names)

    vector = np.concatenate(
        all_features
    ).astype(np.float32)

    if not np.isfinite(vector).all():
        raise FloatingPointError(
            f"NaN/Inf texture feature detected: {path}"
        )

    return (
        vector,
        feature_groups,
        all_names,
    )


# ------------------------------------------------------------
# FEATURE DIMENSION SMOKE CHECK
# ------------------------------------------------------------

(
    example_vector,
    FEATURE_SLICES,
    FEATURE_NAMES,
) = extract_texture_features(
    valid.iloc[0]["resolved_image_path"]
)

print(
    "Total texture feature dimension:",
    len(example_vector),
)

print("\nFeature group slices:")

for group_name, group_slice in (
    FEATURE_SLICES.items()
):
    print(
        f"{group_name}: "
        f"{group_slice.start}–{group_slice.stop} "
        f"({group_slice.stop - group_slice.start} features)"
    )

assert len(example_vector) == 240, (
    "Unexpected texture feature dimension."
)

print("\nTexture feature extraction test passed.")

In [ ]:
# ============================================================
# 11. BUILD OR RESUME MOUTH TEXTURE FEATURE CACHE
# LBP + GLCM + Gabor + Wavelet
# ============================================================

FEATURE_CACHE_ROOT = (
    Path(CONFIG["output_root"])
    / "_mouth_feature_cache"
)

FEATURE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Create a version ID from feature settings
# ------------------------------------------------------------

cache_key_payload = {
    "roi_type": "mouth",
    "image_size": CONFIG["image_size"],
    "lbp_radii": CONFIG["lbp_radii"],
    "lbp_points": CONFIG["lbp_points"],
    "glcm_distances": CONFIG["glcm_distances"],
    "glcm_angles_deg": CONFIG["glcm_angles_deg"],
    "glcm_levels": CONFIG["glcm_levels"],
    "gabor_orientations_deg": CONFIG[
        "gabor_orientations_deg"
    ],
    "gabor_frequencies": CONFIG[
        "gabor_frequencies"
    ],
    "wavelet": CONFIG["wavelet"],
    "wavelet_level": CONFIG["wavelet_level"],
}


CACHE_VERSION = hashlib.sha256(
    json.dumps(
        cache_key_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]


CACHE_FILE = (
    FEATURE_CACHE_ROOT
    / f"mouth_texture_features_{CACHE_VERSION}.joblib"
)

CACHE_MANIFEST = (
    FEATURE_CACHE_ROOT
    / f"mouth_texture_manifest_{CACHE_VERSION}.csv"
)

FEATURE_NAMES_FILE = (
    FEATURE_CACHE_ROOT
    / f"mouth_feature_names_{CACHE_VERSION}.txt"
)


# ------------------------------------------------------------
# Load an existing compatible cache or start a new one
# ------------------------------------------------------------

if (
    CACHE_FILE.exists()
    and not CONFIG["overwrite_feature_cache"]
):
    texture_cache = joblib.load(
        CACHE_FILE
    )

    print(
        "Compatible mouth feature cache loaded."
    )
else:
    texture_cache = {}

    print(
        "A new mouth feature cache will be created."
    )


cached_sample_ids = {
    str(sample_id)
    for sample_id in texture_cache.keys()
}


pending_rows = valid[
    ~valid[
        CONFIG["sample_id_column"]
    ]
    .astype(str)
    .isin(cached_sample_ids)
].copy()


print(
    f"Existing cached samples: "
    f"{len(texture_cache):,}"
)

print(
    f"Pending samples: "
    f"{len(pending_rows):,}"
)


# ------------------------------------------------------------
# Extract and periodically save features
# ------------------------------------------------------------

cache_errors = []
processed_since_flush = 0


for _, row in tqdm(
    pending_rows.iterrows(),
    total=len(pending_rows),
    desc="Extracting mouth texture features",
):
    sample_id = str(
        row[CONFIG["sample_id_column"]]
    )

    image_path = row[
        "resolved_image_path"
    ]

    try:
        vector, _, _ = (
            extract_texture_features(
                image_path
            )
        )

        texture_cache[
            sample_id
        ] = vector

    except Exception as exception:
        cache_errors.append(
            {
                "sample_id": sample_id,
                "image_path": image_path,
                "error_type": type(
                    exception
                ).__name__,
                "error": str(exception),
            }
        )

    processed_since_flush += 1

    if (
        processed_since_flush
        >= CONFIG[
            "feature_cache_flush_every"
        ]
    ):
        atomic_joblib_dump(
            texture_cache,
            CACHE_FILE,
        )

        processed_since_flush = 0


# Save the completed cache
atomic_joblib_dump(
    texture_cache,
    CACHE_FILE,
)


# ------------------------------------------------------------
# Save extraction errors, if any
# ------------------------------------------------------------

if cache_errors:
    atomic_write_csv(
        pd.DataFrame(cache_errors),
        (
            DIRS["audit"]
            / "mouth_texture_feature_errors.csv"
        ),
    )


# ------------------------------------------------------------
# Keep only samples with successfully cached features
# ------------------------------------------------------------

valid["texture_cached"] = (
    valid[
        CONFIG["sample_id_column"]
    ]
    .astype(str)
    .isin(
        {
            str(sample_id)
            for sample_id
            in texture_cache.keys()
        }
    )
)


feature_manifest = valid[
    valid["texture_cached"]
].copy()


atomic_write_csv(
    feature_manifest,
    CACHE_MANIFEST,
)


atomic_write_text(
    "\n".join(FEATURE_NAMES),
    FEATURE_NAMES_FILE,
)


valid = (
    feature_manifest
    .reset_index(drop=True)
)


if valid.empty:
    raise RuntimeError(
        "Texture feature extraction produced "
        "no valid mouth sample."
    )


# ------------------------------------------------------------
# Final cache quality checks
# ------------------------------------------------------------

assert len(valid) == len(texture_cache), (
    "Feature cache and valid manifest sizes differ."
)

for sample_id in valid[
    CONFIG["sample_id_column"]
].astype(str):
    vector = texture_cache[sample_id]

    assert len(vector) == 240, (
        f"Invalid feature dimension: {sample_id}"
    )

    assert np.isfinite(vector).all(), (
        f"NaN/Inf feature detected: {sample_id}"
    )


print("\nMouth texture feature cache completed.")
print("Cache version:", CACHE_VERSION)
print("Final cached samples:", len(texture_cache))
print("Feature extraction errors:", len(cache_errors))
print("Cache file:", CACHE_FILE)

In [ ]:
# ============================================================
# 12. TRAIN-ONLY TEXTURE FEATURE SCALER
# Validation and test data are not used while fitting.
# ============================================================

train_sample_ids = (
    valid.loc[
        valid[CONFIG["split_column"]] == "train",
        CONFIG["sample_id_column"],
    ]
    .astype(str)
)


train_texture_features = np.stack(
    [
        texture_cache[sample_id]
        for sample_id in train_sample_ids
    ]
)


assert train_texture_features.shape[1] == 240, (
    "Unexpected texture feature dimension."
)

assert np.isfinite(
    train_texture_features
).all(), (
    "NaN or Inf detected in training texture features."
)


texture_scaler = StandardScaler()

texture_scaler.fit(
    train_texture_features
)


SCALER_FILE = (
    DIRS["features"]
    / "mouth_texture_scaler.joblib"
)

atomic_joblib_dump(
    texture_scaler,
    SCALER_FILE,
)


feature_schema = {
    "roi_type": "mouth",
    "fit_split": "train",
    "training_sample_count": int(
        train_texture_features.shape[0]
    ),
    "feature_dimension": int(
        train_texture_features.shape[1]
    ),
    "cache_version": CACHE_VERSION,
    "feature_groups": {
        group_name: [
            group_slice.start,
            group_slice.stop,
        ]
        for group_name, group_slice
        in FEATURE_SLICES.items()
    },
}


atomic_write_json(
    feature_schema,
    (
        DIRS["features"]
        / "mouth_texture_feature_schema.json"
    ),
)


# Verify the saved scaler
reloaded_scaler = joblib.load(
    SCALER_FILE
)

scaled_training_features = (
    reloaded_scaler.transform(
        train_texture_features
    )
)


assert np.isfinite(
    scaled_training_features
).all(), (
    "NaN or Inf detected after scaling."
)


print("Texture scaler fitted successfully.")
print(
    "Scaler fit split: train only"
)
print(
    "Training samples used:",
    train_texture_features.shape[0],
)
print(
    "Feature dimension:",
    train_texture_features.shape[1],
)
print(
    "Mean after scaling:",
    round(
        float(
            scaled_training_features.mean()
        ),
        6,
    ),
)
print(
    "Scaler saved:",
    SCALER_FILE,
)

In [ ]:
# ============================================================
# 13. MOUTH DATASET, TRANSFORMS AND DATALOADERS
# ============================================================

# ImageNet normalization used by pretrained Swin V2-Tiny
imagenet_mean = [
    0.485,
    0.456,
    0.406,
]

imagenet_std = [
    0.229,
    0.224,
    0.225,
]


# Geometric augmentation is not applied because the cached
# texture vector must describe the same version of the image
# that is supplied to the Swin branch.
train_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                CONFIG["image_size"],
                CONFIG["image_size"],
            )
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=imagenet_mean,
            std=imagenet_std,
        ),
    ]
)


eval_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                CONFIG["image_size"],
                CONFIG["image_size"],
            )
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=imagenet_mean,
            std=imagenet_std,
        ),
    ]
)


# The experiment uses only the full-fusion configuration.
MODE_GROUPS = {
    "full": [
        "lbp",
        "glcm",
        "gabor",
        "wavelet",
    ],
}


def select_texture_groups(
    vector: np.ndarray,
    mode: str,
) -> np.ndarray:
    if mode not in MODE_GROUPS:
        raise ValueError(
            f"Unsupported experiment mode: {mode}"
        )

    selected_groups = [
        vector[FEATURE_SLICES[group_name]]
        for group_name in MODE_GROUPS[mode]
    ]

    return np.concatenate(
        selected_groups
    ).astype(np.float32)


class MouthFusionDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        mode: str,
        transform,
        scaler: StandardScaler,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.mode = mode
        self.transform = transform
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(
        self,
        index: int,
    ) -> Dict[str, Any]:
        row = self.dataframe.iloc[index]

        image_path = row[
            "resolved_image_path"
        ]

        sample_id = str(
            row[
                CONFIG["sample_id_column"]
            ]
        )

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image_tensor = self.transform(
                image
            )

        raw_texture = (
            texture_cache[sample_id]
            .reshape(1, -1)
        )

        scaled_texture = (
            self.scaler
            .transform(raw_texture)[0]
            .astype(np.float32)
        )

        selected_texture = (
            select_texture_groups(
                scaled_texture,
                self.mode,
            )
        )

        return {
            "image": image_tensor,

            "texture": torch.from_numpy(
                selected_texture
            ),

            "target": torch.tensor(
                int(row["target"]),
                dtype=torch.float32,
            ),

            "sample_id": sample_id,

            "video_id": str(
                row[
                    CONFIG["video_column"]
                ]
            ),

            "image_path": image_path,
        }


def build_loaders(
    mode: str = "full",
):
    split_dataframes = {
        split: valid[
            valid[
                CONFIG["split_column"]
            ] == split
        ].copy()

        for split in [
            "train",
            "val",
            "test",
        ]
    }

    datasets = {
        "train": MouthFusionDataset(
            split_dataframes["train"],
            mode,
            train_transform,
            texture_scaler,
        ),

        "val": MouthFusionDataset(
            split_dataframes["val"],
            mode,
            eval_transform,
            texture_scaler,
        ),

        "test": MouthFusionDataset(
            split_dataframes["test"],
            mode,
            eval_transform,
            texture_scaler,
        ),
    }

    data_generator = torch.Generator()

    data_generator.manual_seed(
        CONFIG["seed"]
    )

    loaders = {
        "train": DataLoader(
            datasets["train"],
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=CONFIG["num_workers"],
            pin_memory=(
                DEVICE.type == "cuda"
            ),
            generator=data_generator,
        ),

        "val": DataLoader(
            datasets["val"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(
                DEVICE.type == "cuda"
            ),
        ),

        "test": DataLoader(
            datasets["test"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(
                DEVICE.type == "cuda"
            ),
        ),
    }

    return datasets, loaders


# ------------------------------------------------------------
# Dataset and batch shape check
# ------------------------------------------------------------

datasets, loaders = build_loaders(
    CONFIG["active_mode"]
)

example_batch = next(
    iter(loaders["train"])
)


assert example_batch["image"].shape[1:] == (
    3,
    224,
    224,
)

assert example_batch["texture"].shape[1] == 240


print("Mouth dataset and DataLoaders are ready.")

print(
    "Train samples:",
    len(datasets["train"]),
)

print(
    "Validation samples:",
    len(datasets["val"]),
)

print(
    "Test samples:",
    len(datasets["test"]),
)

print(
    "Image batch shape:",
    tuple(
        example_batch["image"].shape
    ),
)

print(
    "Texture batch shape:",
    tuple(
        example_batch["texture"].shape
    ),
)

print(
    "Target batch shape:",
    tuple(
        example_batch["target"].shape
    ),
)

In [ ]:
# ============================================================
# 14. SWIN V2-TINY + TEXTURE FUSION MODEL
# Full mode: Swin + LBP + GLCM + Gabor + Wavelet
# ============================================================

class SwinTextureFusion(nn.Module):
    def __init__(
        self,
        texture_dim: int,
        hidden_dim: int,
        dropout: float,
        pretrained: bool = True,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Deep visual branch: Swin V2-Tiny
        # ----------------------------------------------------

        selected_weights = (
            Swin_V2_T_Weights.DEFAULT
            if pretrained
            else None
        )

        self.backbone = swin_v2_t(
            weights=selected_weights
        )

        self.swin_dim = (
            self.backbone.head.in_features
        )

        # Remove the original ImageNet classifier
        self.backbone.head = nn.Identity()


        self.swin_projection = nn.Sequential(
            nn.LayerNorm(
                self.swin_dim
            ),

            nn.Linear(
                self.swin_dim,
                hidden_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),
        )


        # ----------------------------------------------------
        # Handcrafted texture branch
        # ----------------------------------------------------

        self.texture_dim = texture_dim

        self.texture_projection = nn.Sequential(
            nn.LayerNorm(
                texture_dim
            ),

            nn.Linear(
                texture_dim,
                hidden_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),
        )


        # ----------------------------------------------------
        # Feature-level fusion and binary classifier
        # ----------------------------------------------------

        fusion_dim = (
            hidden_dim * 2
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                fusion_dim,
                hidden_dim,
            ),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim // 2,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim // 2,
                1,
            ),
        )


    def forward(
        self,
        image: torch.Tensor,
        texture: torch.Tensor,
    ) -> torch.Tensor:
        deep_features = self.backbone(
            image
        )

        deep_projection = (
            self.swin_projection(
                deep_features
            )
        )

        texture_projection = (
            self.texture_projection(
                texture
            )
        )

        fused_features = torch.cat(
            [
                deep_projection,
                texture_projection,
            ],
            dim=1,
        )

        logits = self.classifier(
            fused_features
        )

        return logits.squeeze(1)


    def freeze_backbone(
        self,
    ) -> None:
        for parameter in (
            self.backbone.parameters()
        ):
            parameter.requires_grad = False


    def unfreeze_backbone_tail(
        self,
    ) -> None:
        # Keep the whole backbone frozen first.
        for parameter in (
            self.backbone.parameters()
        ):
            parameter.requires_grad = False

        # Unfreeze the final Swin feature stage.
        for parameter in (
            self.backbone
            .features[-1]
            .parameters()
        ):
            parameter.requires_grad = True

        # Unfreeze the final normalization layer.
        for parameter in (
            self.backbone
            .norm
            .parameters()
        ):
            parameter.requires_grad = True

        # The projection layer remains trainable.
        for parameter in (
            self.swin_projection
            .parameters()
        ):
            parameter.requires_grad = True


TEXTURE_DIMENSION = len(
    select_texture_groups(
        np.zeros(
            len(FEATURE_NAMES),
            dtype=np.float32,
        ),
        CONFIG["active_mode"],
    )
)


assert TEXTURE_DIMENSION == 240


print("Swin V2-Tiny texture-fusion model definition is ready.")
print("Texture input dimension:", TEXTURE_DIMENSION)
print("Fusion hidden dimension:", CONFIG["hidden_dim"])
print("Active experiment mode:", CONFIG["active_mode"])

In [ ]:
# ============================================================
# 15. METRICS, THRESHOLD SEARCH, OPTIMIZER AND CHECKPOINT
# ============================================================

def binary_metrics(
    targets: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    predictions = (
        probabilities >= threshold
    ).astype(int)

    results = {
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                targets,
                predictions,
            ),

        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            targets,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
    }

    if len(np.unique(targets)) == 2:
        results["roc_auc"] = (
            roc_auc_score(
                targets,
                probabilities,
            )
        )

        results["average_precision"] = (
            average_precision_score(
                targets,
                probabilities,
            )
        )
    else:
        results["roc_auc"] = float("nan")
        results["average_precision"] = (
            float("nan")
        )

    return {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in results.items()
    }


def find_best_threshold(
    targets: np.ndarray,
    probabilities: np.ndarray,
) -> Tuple[float, float]:
    thresholds = np.linspace(
        CONFIG["threshold_search_min"],
        CONFIG["threshold_search_max"],
        CONFIG["threshold_search_steps"],
    )

    best_threshold = CONFIG[
        "threshold"
    ]

    best_f1_score = -np.inf

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype(int)

        current_f1 = f1_score(
            targets,
            predictions,
            zero_division=0,
        )

        if current_f1 > best_f1_score:
            best_f1_score = float(
                current_f1
            )

            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        best_f1_score,
    )


def build_optimizer(
    model: SwinTextureFusion,
):
    backbone_parameters = []
    head_parameters = []

    for parameter_name, parameter in (
        model.named_parameters()
    ):
        if not parameter.requires_grad:
            continue

        if parameter_name.startswith(
            "backbone."
        ):
            backbone_parameters.append(
                parameter
            )
        else:
            head_parameters.append(
                parameter
            )

    parameter_groups = []

    if backbone_parameters:
        parameter_groups.append(
            {
                "params": backbone_parameters,
                "lr": CONFIG[
                    "backbone_learning_rate"
                ],
            }
        )

    if head_parameters:
        parameter_groups.append(
            {
                "params": head_parameters,
                "lr": CONFIG[
                    "learning_rate"
                ],
            }
        )

    if not parameter_groups:
        raise RuntimeError(
            "No trainable model parameter found."
        )

    return torch.optim.AdamW(
        parameter_groups,
        weight_decay=CONFIG[
            "weight_decay"
        ],
    )


def capture_rng_state() -> Dict[str, Any]:
    rng_state = {
        "python_random_state":
            random.getstate(),

        "numpy_random_state":
            np.random.get_state(),

        "torch_rng_state":
            torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        rng_state["cuda_rng_state"] = (
            torch.cuda.get_rng_state_all()
        )

    return rng_state


def checkpoint_state(
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    gradient_scaler,
    best_metric: float,
    best_threshold: float,
    history: List[Dict[str, float]],
) -> Dict[str, Any]:
    state = {
        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            gradient_scaler.state_dict(),

        "best_metric_score":
            float(best_metric),

        "best_threshold":
            float(best_threshold),

        "selection_metric":
            "val_roc_auc",

        "history":
            history,

        "config":
            CONFIG,

        "mode":
            CONFIG["active_mode"],

        "roi_type":
            "mouth",

        "feature_slices": {
            group_name: [
                group_slice.start,
                group_slice.stop,
            ]

            for group_name, group_slice
            in FEATURE_SLICES.items()
        },
    }

    state.update(
        capture_rng_state()
    )

    return state


# ------------------------------------------------------------
# Small metric-function verification
# ------------------------------------------------------------

test_targets = np.asarray(
    [0, 0, 1, 1],
    dtype=np.int64,
)

test_probabilities = np.asarray(
    [0.1, 0.4, 0.6, 0.9],
    dtype=np.float64,
)

metric_test = binary_metrics(
    test_targets,
    test_probabilities,
    threshold=0.50,
)

assert metric_test["accuracy"] == 1.0
assert metric_test["f1"] == 1.0

threshold_test, threshold_f1_test = (
    find_best_threshold(
        test_targets,
        test_probabilities,
    )
)

assert 0.10 <= threshold_test <= 0.90
assert threshold_f1_test == 1.0


print(
    "Metrics, threshold search, optimizer and "
    "checkpoint helpers are ready."
)
print(
    "Metric verification accuracy:",
    metric_test["accuracy"],
)
print(
    "Threshold search verification:",
    threshold_test,
)

In [ ]:
# ============================================================
# 16. TRAINING, VALIDATION AND PREDICTION LOOPS
# ============================================================

def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer=None,
    gradient_scaler=None,
    training: bool = False,
    max_batches: Optional[int] = None,
) -> Tuple[
    float,
    np.ndarray,
    np.ndarray,
]:
    model.train(training)

    losses = []
    all_targets = []
    all_probabilities = []

    for batch_index, batch in enumerate(
        loader
    ):
        if (
            max_batches is not None
            and batch_index >= max_batches
        ):
            break

        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        textures = batch["texture"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        if training:
            if optimizer is None:
                raise ValueError(
                    "Optimizer is required for training."
                )

            if gradient_scaler is None:
                raise ValueError(
                    "GradientScaler is required for training."
                )

            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):
            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(
                    CONFIG["amp"]
                    and DEVICE.type == "cuda"
                ),
            ):
                logits = model(
                    images,
                    textures,
                )

                loss = criterion(
                    logits,
                    targets,
                )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"NaN or Inf loss detected "
                f"at batch {batch_index}."
            )

        if training:
            gradient_scaler.scale(
                loss
            ).backward()

            gradient_scaler.unscale_(
                optimizer
            )

            gradient_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    model.parameters(),
                    CONFIG[
                        "gradient_clip_norm"
                    ],
                )
            )

            if not torch.isfinite(
                gradient_norm
            ):
                raise FloatingPointError(
                    "NaN or Inf gradient detected."
                )

            gradient_scaler.step(
                optimizer
            )

            gradient_scaler.update()

        losses.append(
            float(
                loss.detach().cpu()
            )
        )

        probabilities = (
            torch.sigmoid(logits)
            .detach()
            .cpu()
            .numpy()
        )

        all_probabilities.extend(
            probabilities.tolist()
        )

        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    if not losses:
        raise RuntimeError(
            "The DataLoader produced no batch."
        )

    return (
        float(np.mean(losses)),

        np.asarray(
            all_targets,
            dtype=np.int64,
        ),

        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
    )


def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    threshold: float,
) -> pd.DataFrame:
    model.eval()

    prediction_records = []

    with torch.no_grad():
        for batch in tqdm(
            loader,
            desc="Predicting test samples",
        ):
            images = batch["image"].to(
                DEVICE,
                non_blocking=True,
            )

            textures = batch["texture"].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(
                    CONFIG["amp"]
                    and DEVICE.type == "cuda"
                ),
            ):
                logits = model(
                    images,
                    textures,
                )

            probabilities = (
                torch.sigmoid(logits)
                .cpu()
                .numpy()
            )

            targets = (
                batch["target"]
                .numpy()
            )

            for sample_index in range(
                len(probabilities)
            ):
                probability_fake = float(
                    probabilities[
                        sample_index
                    ]
                )

                target = int(
                    targets[
                        sample_index
                    ]
                )

                prediction = int(
                    probability_fake
                    >= threshold
                )

                prediction_records.append(
                    {
                        "sample_id":
                            batch[
                                "sample_id"
                            ][sample_index],

                        "video_id":
                            batch[
                                "video_id"
                            ][sample_index],

                        "image_path":
                            batch[
                                "image_path"
                            ][sample_index],

                        "target":
                            target,

                        "probability_fake":
                            probability_fake,

                        "prediction":
                            prediction,

                        "correct":
                            prediction == target,
                    }
                )

    predictions = pd.DataFrame(
        prediction_records
    )

    if predictions.empty:
        raise RuntimeError(
            "No test prediction was produced."
        )

    return predictions


print(
    "Training, validation and prediction "
    "loops are ready."
)

In [ ]:
# ============================================================
# 17. SMOKE TEST
# Full training is blocked if this test fails.
# ============================================================

def smoke_test() -> None:
    print("Starting smoke test...")

    smoke_datasets, smoke_loaders = (
        build_loaders(
            CONFIG["active_mode"]
        )
    )

    model = SwinTextureFusion(
        texture_dim=TEXTURE_DIMENSION,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        pretrained=CONFIG["pretrained"],
    ).to(DEVICE)


    # Same training strategy as the full experiment:
    # freeze the Swin backbone initially.
    if CONFIG["freeze_backbone_epochs"] > 0:
        model.freeze_backbone()


    optimizer = build_optimizer(
        model
    )


    train_targets = (
        smoke_datasets["train"]
        .dataframe["target"]
        .to_numpy()
    )


    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.asarray([0, 1]),
        y=train_targets,
    )


    positive_weight = torch.tensor(
        [
            class_weights[1]
            / class_weights[0]
        ],
        device=DEVICE,
        dtype=torch.float32,
    )


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )


    gradient_scaler = (
        torch.amp.GradScaler(
            "cuda",
            enabled=(
                CONFIG["amp"]
                and DEVICE.type == "cuda"
            ),
        )
    )


    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                CONFIG["epochs"],
            ),
        )
    )


    train_loss, train_y, train_p = (
        run_epoch(
            model=model,
            loader=smoke_loaders["train"],
            criterion=criterion,
            optimizer=optimizer,
            gradient_scaler=gradient_scaler,
            training=True,
            max_batches=CONFIG[
                "smoke_train_batches"
            ],
        )
    )


    validation_loss, val_y, val_p = (
        run_epoch(
            model=model,
            loader=smoke_loaders["val"],
            criterion=criterion,
            training=False,
            max_batches=CONFIG[
                "smoke_val_batches"
            ],
        )
    )


    assert np.isfinite(train_loss)
    assert np.isfinite(validation_loss)
    assert np.isfinite(train_p).all()
    assert np.isfinite(val_p).all()


    smoke_checkpoint_path = (
        DIRS["checkpoints"]
        / "smoke_test.ckpt"
    )


    smoke_state = checkpoint_state(
        epoch=0,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        gradient_scaler=gradient_scaler,
        best_metric=0.0,
        best_threshold=CONFIG[
            "threshold"
        ],
        history=[],
    )


    atomic_torch_save(
        smoke_state,
        smoke_checkpoint_path,
    )


    reloaded_checkpoint = torch.load(
        smoke_checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )


    required_checkpoint_keys = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "best_metric_score",
        "best_threshold",
        "config",
    }


    assert required_checkpoint_keys.issubset(
        reloaded_checkpoint.keys()
    )


    # Smoke checkpoint is temporary.
    smoke_checkpoint_path.unlink(
        missing_ok=True
    )


    print("\nSmoke test passed.")
    print(
        "Training batches tested:",
        CONFIG["smoke_train_batches"],
    )
    print(
        "Validation batches tested:",
        CONFIG["smoke_val_batches"],
    )
    print(
        "Smoke training loss:",
        round(train_loss, 6),
    )
    print(
        "Smoke validation loss:",
        round(validation_loss, 6),
    )
    print(
        "Checkpoint save/reload:",
        "PASSED",
    )


    del (
        model,
        optimizer,
        scheduler,
        gradient_scaler,
        smoke_loaders,
        smoke_datasets,
        reloaded_checkpoint,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if CONFIG["smoke_test"]:
    smoke_test()
else:
    raise RuntimeError(
        "Smoke test must be enabled "
        "before full training."
    )

In [ ]:
# ============================================================
# 18. FULL TRAINING AND TEST EVALUATION FUNCTION
# Best checkpoint: Validation ROC-AUC
# Test threshold: Best validation F1 threshold
# ============================================================

MODE_DIR = (
    RUN_DIR
    / CONFIG["active_mode"]
)

MODE_DIRS = {
    "root": MODE_DIR,

    "checkpoints":
        MODE_DIR / "checkpoints",

    "metrics":
        MODE_DIR / "metrics",

    "predictions":
        MODE_DIR / "predictions",

    "figures":
        MODE_DIR / "figures",
}


for directory in MODE_DIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def train_full_experiment() -> Dict[str, Any]:
    print("=" * 70)
    print("Training mode: full")
    print("ROI type: mouth")
    print("=" * 70)


    experiment_datasets, experiment_loaders = (
        build_loaders(
            CONFIG["active_mode"]
        )
    )


    model = SwinTextureFusion(
        texture_dim=TEXTURE_DIMENSION,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
        pretrained=CONFIG["pretrained"],
    ).to(DEVICE)


    if CONFIG["freeze_backbone_epochs"] > 0:
        model.freeze_backbone()

        print(
            "Swin backbone frozen for "
            f"the first "
            f"{CONFIG['freeze_backbone_epochs']} epochs."
        )


    train_targets = (
        experiment_datasets["train"]
        .dataframe["target"]
        .to_numpy()
    )


    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.asarray([0, 1]),
        y=train_targets,
    )


    positive_weight = torch.tensor(
        [
            class_weights[1]
            / class_weights[0]
        ],
        device=DEVICE,
        dtype=torch.float32,
    )


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=positive_weight
    )


    optimizer = build_optimizer(
        model
    )


    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                CONFIG["epochs"],
            ),
        )
    )


    gradient_scaler = (
        torch.amp.GradScaler(
            "cuda",
            enabled=(
                CONFIG["amp"]
                and DEVICE.type == "cuda"
            ),
        )
    )


    history = []

    best_validation_auc = -np.inf

    best_validation_threshold = (
        CONFIG["threshold"]
    )

    patience_counter = 0


    for epoch_index in range(
        CONFIG["epochs"]
    ):
        epoch_number = (
            epoch_index + 1
        )


        # ----------------------------------------------------
        # Unfreeze final Swin stage after the first 5 epochs
        # ----------------------------------------------------

        if (
            CONFIG["unfreeze_last_stages"]
            and epoch_index
            == CONFIG[
                "freeze_backbone_epochs"
            ]
        ):
            model.unfreeze_backbone_tail()

            optimizer = build_optimizer(
                model
            )

            scheduler = (
                torch.optim.lr_scheduler
                .CosineAnnealingLR(
                    optimizer,
                    T_max=max(
                        1,
                        CONFIG["epochs"]
                        - epoch_index,
                    ),
                )
            )

            print(
                "\nFinal Swin stage unfrozen."
            )


        # ----------------------------------------------------
        # Training epoch
        # ----------------------------------------------------

        (
            train_loss,
            train_y,
            train_probabilities,
        ) = run_epoch(
            model=model,
            loader=experiment_loaders[
                "train"
            ],
            criterion=criterion,
            optimizer=optimizer,
            gradient_scaler=gradient_scaler,
            training=True,
        )


        # ----------------------------------------------------
        # Validation epoch
        # ----------------------------------------------------

        (
            validation_loss,
            validation_y,
            validation_probabilities,
        ) = run_epoch(
            model=model,
            loader=experiment_loaders[
                "val"
            ],
            criterion=criterion,
            training=False,
        )


        # Training metrics use the configured 0.50 threshold.
        train_metrics = binary_metrics(
            train_y,
            train_probabilities,
            CONFIG["threshold"],
        )


        # Validation threshold is selected only from
        # validation predictions.
        (
            epoch_threshold,
            epoch_threshold_f1,
        ) = find_best_threshold(
            validation_y,
            validation_probabilities,
        )


        validation_metrics = (
            binary_metrics(
                validation_y,
                validation_probabilities,
                epoch_threshold,
            )
        )


        current_validation_auc = (
            validation_metrics[
                "roc_auc"
            ]
        )


        epoch_record = {
            "epoch":
                epoch_number,

            "train_loss":
                train_loss,

            "val_loss":
                validation_loss,

            "validation_threshold":
                epoch_threshold,

            "validation_threshold_f1":
                epoch_threshold_f1,

            **{
                f"train_{metric_name}":
                    metric_value

                for metric_name, metric_value
                in train_metrics.items()
            },

            **{
                f"val_{metric_name}":
                    metric_value

                for metric_name, metric_value
                in validation_metrics.items()
            },

            "learning_rate":
                optimizer.param_groups[0][
                    "lr"
                ],
        }


        history.append(
            epoch_record
        )


        atomic_write_csv(
            pd.DataFrame(history),
            (
                MODE_DIRS["metrics"]
                / "epoch_metrics.csv"
            ),
        )


        # ----------------------------------------------------
        # Best checkpoint selection by validation ROC-AUC
        # ----------------------------------------------------

        improved = (
            current_validation_auc
            > best_validation_auc
        )


        if improved:
            best_validation_auc = float(
                current_validation_auc
            )

            best_validation_threshold = float(
                epoch_threshold
            )

            patience_counter = 0

        else:
            patience_counter += 1


        checkpoint = checkpoint_state(
            epoch=epoch_number,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            gradient_scaler=gradient_scaler,
            best_metric=best_validation_auc,
            best_threshold=(
                best_validation_threshold
            ),
            history=history,
        )


        # Latest recoverable state
        atomic_torch_save(
            checkpoint,
            (
                MODE_DIRS["checkpoints"]
                / "last.ckpt"
            ),
        )


        # Best validation state
        if improved:
            atomic_torch_save(
                checkpoint,
                (
                    MODE_DIRS[
                        "checkpoints"
                    ]
                    / "best.ckpt"
                ),
            )


        scheduler.step()


        print(
            f"Epoch {epoch_number:02d}/"
            f"{CONFIG['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {validation_loss:.4f} | "
            f"Val F1: "
            f"{validation_metrics['f1']:.4f} | "
            f"Val AUC: "
            f"{current_validation_auc:.4f} | "
            f"Threshold: "
            f"{epoch_threshold:.3f} | "
            f"Patience: "
            f"{patience_counter}/"
            f"{CONFIG['patience']}"
        )


        if (
            patience_counter
            >= CONFIG["patience"]
        ):
            print(
                "\nEarly stopping activated."
            )

            break


    # --------------------------------------------------------
    # Load best validation checkpoint
    # --------------------------------------------------------

    best_checkpoint_path = (
        MODE_DIRS["checkpoints"]
        / "best.ckpt"
    )


    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "Best checkpoint was not created."
        )


    best_checkpoint = torch.load(
        best_checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )


    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )


    selected_threshold = float(
        best_checkpoint[
            "best_threshold"
        ]
    )


    best_epoch = int(
        best_checkpoint["epoch"]
    )


    # --------------------------------------------------------
    # Final test evaluation
    # --------------------------------------------------------

    test_predictions = predict_loader(
        model=model,
        loader=experiment_loaders["test"],
        threshold=selected_threshold,
    )


    test_metrics = binary_metrics(
        test_predictions[
            "target"
        ].to_numpy(),

        test_predictions[
            "probability_fake"
        ].to_numpy(),

        selected_threshold,
    )


    test_metrics.update(
        {
            "best_epoch":
                best_epoch,

            "selected_threshold":
                selected_threshold,

            "best_validation_roc_auc":
                float(
                    best_checkpoint[
                        "best_metric_score"
                    ]
                ),

            "test_sample_count":
                int(
                    len(test_predictions)
                ),

            "mode":
                CONFIG["active_mode"],

            "roi_type":
                "mouth",
        }
    )


    atomic_write_csv(
        test_predictions,
        (
            MODE_DIRS[
                "predictions"
            ]
            / "test_predictions.csv"
        ),
    )


    atomic_write_json(
        test_metrics,
        (
            MODE_DIRS["metrics"]
            / "test_metrics.json"
        ),
    )


    experiment_summary = {
        "mode":
            CONFIG["active_mode"],

        "roi_type":
            "mouth",

        "best_epoch":
            best_epoch,

        "selected_threshold":
            selected_threshold,

        "accuracy":
            test_metrics["accuracy"],

        "balanced_accuracy":
            test_metrics[
                "balanced_accuracy"
            ],

        "precision":
            test_metrics["precision"],

        "recall":
            test_metrics["recall"],

        "f1":
            test_metrics["f1"],

        "roc_auc":
            test_metrics["roc_auc"],

        "average_precision":
            test_metrics[
                "average_precision"
            ],
    }


    print("\n" + "=" * 70)
    print("Full training completed.")
    print("Best epoch:", best_epoch)
    print(
        "Selected validation threshold:",
        round(
            selected_threshold,
            4,
        ),
    )
    print("\nTest metrics:")

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
    ]:
        print(
            f"{metric_name}: "
            f"{test_metrics[metric_name]:.4f}"
        )

    print("=" * 70)


    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return experiment_summary


print(
    "Full mouth experiment training "
    "function is ready."
)
print(
    "Training has not started yet."
)

In [ ]:
# ============================================================
# 19. RUN THE FULL MOUTH EXPERIMENT
# Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet
# ============================================================

experiment_result = (
    train_full_experiment()
)


experiment_result_dataframe = (
    pd.DataFrame(
        [experiment_result]
    )
)


atomic_write_csv(
    experiment_result_dataframe,
    (
        DIRS["metrics"]
        / "full_experiment_summary.csv"
    ),
)


atomic_write_json(
    experiment_result,
    (
        DIRS["metrics"]
        / "full_experiment_summary.json"
    ),
)


print("\nExperiment summary saved.")

display(
    experiment_result_dataframe
)

In [ ]:
# ============================================================
# FIX: AMP-SAFE TRAINING LOOP
# GradScaler handles transient non-finite gradients itself.
# Model architecture and hyperparameters are unchanged.
# ============================================================

def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer=None,
    gradient_scaler=None,
    training: bool = False,
    max_batches: Optional[int] = None,
) -> Tuple[
    float,
    np.ndarray,
    np.ndarray,
]:
    model.train(training)

    losses = []
    all_targets = []
    all_probabilities = []

    for batch_index, batch in enumerate(loader):
        if (
            max_batches is not None
            and batch_index >= max_batches
        ):
            break

        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        textures = batch["texture"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(
                    CONFIG["amp"]
                    and DEVICE.type == "cuda"
                ),
            ):
                logits = model(
                    images,
                    textures,
                )

                loss = criterion(
                    logits,
                    targets,
                )

        # Loss must always remain finite.
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"NaN or Inf loss detected "
                f"at batch {batch_index}."
            )

        if training:
            gradient_scaler.scale(
                loss
            ).backward()

            gradient_scaler.unscale_(
                optimizer
            )

            # Same behavior as the friend's original notebook.
            # GradScaler detects overflow during scaler.step()
            # and safely skips that optimizer update.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CONFIG[
                    "gradient_clip_norm"
                ],
                error_if_nonfinite=False,
            )

            gradient_scaler.step(
                optimizer
            )

            gradient_scaler.update()

        losses.append(
            float(
                loss.detach().cpu()
            )
        )

        probabilities = (
            torch.sigmoid(logits)
            .detach()
            .cpu()
            .numpy()
        )

        all_probabilities.extend(
            probabilities.tolist()
        )

        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    if not losses:
        raise RuntimeError(
            "The DataLoader produced no batch."
        )

    return (
        float(np.mean(losses)),

        np.asarray(
            all_targets,
            dtype=np.int64,
        ),

        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
    )


print("AMP-safe training loop updated successfully.")
print("Model architecture and hyperparameters were not changed.")

In [ ]:
# ============================================================
# 20. MAIN EXPERIMENT FIGURES
# English labels and minimum 600-pixel short edge
# ============================================================

history = pd.read_csv(
    MODE_DIRS["metrics"]
    / "epoch_metrics.csv"
)

test_predictions = pd.read_csv(
    MODE_DIRS["predictions"]
    / "test_predictions.csv"
)

with open(
    MODE_DIRS["metrics"]
    / "test_metrics.json",
    "r",
    encoding="utf-8",
) as file:
    saved_test_metrics = json.load(file)


selected_threshold = float(
    saved_test_metrics[
        "selected_threshold"
    ]
)


def save_mode_figure(
    figure: plt.Figure,
    filename: str,
) -> Path:
    target = (
        MODE_DIRS["figures"]
        / filename
    )

    figure.savefig(
        target,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(figure)

    with Image.open(target) as image:
        if min(image.size) < 600:
            raise AssertionError(
                f"Figure resolution is insufficient: "
                f"{image.size}"
            )

    return target


# ------------------------------------------------------------
# 1. Training and validation loss
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history["epoch"],
    history["train_loss"],
    label="Training Loss",
    color="#3182CE",
    linewidth=2,
)

axis.plot(
    history["epoch"],
    history["val_loss"],
    label="Validation Loss",
    color="#DD6B20",
    linewidth=2,
    linestyle="--",
)

axis.set_title(
    "Training and Validation Loss Curve",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Epoch",
    fontsize=11,
)

axis.set_ylabel(
    "Loss",
    fontsize=11,
)

axis.legend(
    frameon=True,
    loc="upper right",
)

axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "01_training_validation_loss_curve.png",
)


# ------------------------------------------------------------
# 2. Training and validation F1
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history["epoch"],
    history["train_f1"],
    label="Training F1",
    color="#3182CE",
    linewidth=2,
)

axis.plot(
    history["epoch"],
    history["val_f1"],
    label="Validation F1",
    color="#DD6B20",
    linewidth=2,
    linestyle="--",
)

axis.set_title(
    "Training and Validation F1 Curve",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Epoch",
    fontsize=11,
)

axis.set_ylabel(
    "F1 Score",
    fontsize=11,
)

axis.set_ylim(
    0,
    1,
)

axis.legend(
    frameon=True,
    loc="lower right",
)

axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "02_training_validation_f1_curve.png",
)


# ------------------------------------------------------------
# 3. Test confusion matrix
# ------------------------------------------------------------

confusion = confusion_matrix(
    test_predictions["target"],
    test_predictions["prediction"],
    labels=[0, 1],
)

figure, axis = plt.subplots(
    figsize=(8, 8),
    dpi=150,
)

matrix_image = axis.imshow(
    confusion,
    cmap="Blues",
    vmin=0,
    vmax=confusion.max(),
)

for row_index in range(2):
    for column_index in range(2):
        cell_value = confusion[
            row_index,
            column_index,
        ]

        text_color = (
            "white"
            if cell_value
            > confusion.max() / 2
            else "black"
        )

        axis.text(
            column_index,
            row_index,
            str(cell_value),
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            color=text_color,
        )

axis.set_xticks(
    [0, 1],
    labels=["Real", "Fake"],
)

axis.set_yticks(
    [0, 1],
    labels=["Real", "Fake"],
)

axis.set_title(
    "Test Confusion Matrix",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Predicted Class",
    fontsize=11,
)

axis.set_ylabel(
    "True Class",
    fontsize=11,
)

figure.colorbar(
    matrix_image,
    ax=axis,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "03_test_confusion_matrix.png",
)


# ------------------------------------------------------------
# 4. Test ROC curve
# ------------------------------------------------------------

false_positive_rate, true_positive_rate, _ = (
    roc_curve(
        test_predictions["target"],
        test_predictions[
            "probability_fake"
        ],
    )
)

test_auc = roc_auc_score(
    test_predictions["target"],
    test_predictions[
        "probability_fake"
    ],
)

figure, axis = plt.subplots(
    figsize=(9, 7),
    dpi=150,
)

axis.plot(
    false_positive_rate,
    true_positive_rate,
    color="#3182CE",
    linewidth=2,
    label=(
        f"ROC AUC = "
        f"{test_auc:.4f}"
    ),
)

axis.plot(
    [0, 1],
    [0, 1],
    color="#718096",
    linestyle="--",
    label="Random Classifier",
)

axis.set_title(
    "Test Receiver Operating Characteristic Curve",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "False Positive Rate",
    fontsize=11,
)

axis.set_ylabel(
    "True Positive Rate",
    fontsize=11,
)

axis.legend(
    frameon=True,
    loc="lower right",
)

axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "04_test_roc_curve.png",
)


# ------------------------------------------------------------
# 5. Test precision-recall curve
# ------------------------------------------------------------

precision_values, recall_values, _ = (
    precision_recall_curve(
        test_predictions["target"],
        test_predictions[
            "probability_fake"
        ],
    )
)

average_precision = (
    average_precision_score(
        test_predictions["target"],
        test_predictions[
            "probability_fake"
        ],
    )
)

figure, axis = plt.subplots(
    figsize=(9, 7),
    dpi=150,
)

axis.plot(
    recall_values,
    precision_values,
    color="#DD6B20",
    linewidth=2,
    label=(
        f"Average Precision = "
        f"{average_precision:.4f}"
    ),
)

axis.set_title(
    "Test Precision-Recall Curve",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Recall",
    fontsize=11,
)

axis.set_ylabel(
    "Precision",
    fontsize=11,
)

axis.set_xlim(
    0,
    1,
)

axis.set_ylim(
    0,
    1.05,
)

axis.legend(
    frameon=True,
    loc="lower left",
)

axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "05_test_precision_recall_curve.png",
)


# ------------------------------------------------------------
# 6. Prediction probability distribution
# ------------------------------------------------------------

real_probabilities = test_predictions.loc[
    test_predictions["target"] == 0,
    "probability_fake",
]

fake_probabilities = test_predictions.loc[
    test_predictions["target"] == 1,
    "probability_fake",
]

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.hist(
    real_probabilities,
    bins=20,
    alpha=0.65,
    color="#3182CE",
    label="Real Samples",
)

axis.hist(
    fake_probabilities,
    bins=20,
    alpha=0.65,
    color="#DD6B20",
    label="Fake Samples",
)

axis.axvline(
    selected_threshold,
    color="#2D3748",
    linestyle="--",
    linewidth=2,
    label=(
        f"Selected Threshold = "
        f"{selected_threshold:.3f}"
    ),
)

axis.set_title(
    "Distribution of Predicted Fake Probabilities",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

axis.set_xlabel(
    "Predicted Probability of Fake Class",
    fontsize=11,
)

axis.set_ylabel(
    "Number of Samples",
    fontsize=11,
)

axis.legend(
    frameon=True,
)

axis.grid(
    axis="y",
    alpha=0.25,
)

figure.tight_layout()

save_mode_figure(
    figure,
    "06_test_probability_distribution.png",
)


figure_files = sorted(
    MODE_DIRS["figures"].glob("*.png")
)

print("Experiment figures saved successfully.")
print("Number of figures:", len(figure_files))

for figure_path in figure_files:
    with Image.open(figure_path) as image:
        print(
            figure_path.name,
            "->",
            image.size,
        )

In [ ]:
print("Colab connection restored.")

try:
    print("Run directory:", RUN_DIR)
    print("Variables are still available.")
except NameError:
    print("Runtime variables were cleared.")

## Yerel kaynak hazırlığı\n\nBu arşivlenmiş proje paketinde dış kaynak bağlantısı kullanılmaz. Gerekli Deney 1 dosyaları repo içindeki yerel kaynak çıktıları, manifestler ve açıklama dosyalarıyla temsil edilir.

In [ ]:
# ============================================================
# 21. VIDEO-ID-LEVEL EVALUATION
# Same fixed 0.50 threshold as the eye experiment
# ============================================================

import os
import io

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def atomic_text_write(
    text: str,
    target: Path,
) -> None:
    target = Path(target)

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = target.with_suffix(
        target.suffix + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(text)
        file.flush()
        os.fsync(file.fileno())

    os.replace(
        temporary,
        target,
    )


test_predictions = pd.read_csv(
    MODE_DIRS["predictions"]
    / "test_predictions.csv"
)


VIDEO_THRESHOLD = 0.50


video_predictions = (
    test_predictions
    .groupby(
        "video_id",
        as_index=False,
    )
    .agg(
        target=(
            "target",
            "first",
        ),

        probability_fake=(
            "probability_fake",
            "mean",
        ),

        sample_count=(
            "sample_id",
            "count",
        ),
    )
)


video_predictions["prediction"] = (
    video_predictions[
        "probability_fake"
    ] >= VIDEO_THRESHOLD
).astype(int)


video_predictions["correct"] = (
    video_predictions["prediction"]
    == video_predictions["target"]
)


video_targets = (
    video_predictions["target"]
    .to_numpy()
)

video_probabilities = (
    video_predictions[
        "probability_fake"
    ]
    .to_numpy()
)

video_labels = (
    video_predictions["prediction"]
    .to_numpy()
)


video_metrics = {
    "threshold":
        VIDEO_THRESHOLD,

    "video_id_count":
        int(
            len(video_predictions)
        ),

    "accuracy":
        float(
            accuracy_score(
                video_targets,
                video_labels,
            )
        ),

    "balanced_accuracy":
        float(
            balanced_accuracy_score(
                video_targets,
                video_labels,
            )
        ),

    "precision":
        float(
            precision_score(
                video_targets,
                video_labels,
                zero_division=0,
            )
        ),

    "recall":
        float(
            recall_score(
                video_targets,
                video_labels,
                zero_division=0,
            )
        ),

    "f1":
        float(
            f1_score(
                video_targets,
                video_labels,
                zero_division=0,
            )
        ),
}


if len(
    np.unique(video_targets)
) == 2:
    video_metrics["roc_auc"] = float(
        roc_auc_score(
            video_targets,
            video_probabilities,
        )
    )

    video_metrics[
        "average_precision"
    ] = float(
        average_precision_score(
            video_targets,
            video_probabilities,
        )
    )


video_csv_buffer = io.StringIO()

video_predictions.to_csv(
    video_csv_buffer,
    index=False,
)

atomic_text_write(
    video_csv_buffer.getvalue(),
    (
        MODE_DIRS["predictions"]
        / "video_level_predictions.csv"
    ),
)


atomic_text_write(
    json.dumps(
        video_metrics,
        indent=2,
        ensure_ascii=False,
    ),
    (
        MODE_DIRS["metrics"]
        / "video_level_metrics.json"
    ),
)


print("Video-ID-level evaluation completed.")
print("Fixed threshold:", VIDEO_THRESHOLD)
print(
    "Number of video IDs:",
    len(video_predictions),
)
print(
    "Minimum samples per video ID:",
    int(
        video_predictions[
            "sample_count"
        ].min()
    ),
)
print(
    "Maximum samples per video ID:",
    int(
        video_predictions[
            "sample_count"
        ].max()
    ),
)

print("\nVideo-ID-level metrics:")

for metric_name, metric_value in (
    video_metrics.items()
):
    print(
        f"{metric_name}: {metric_value}"
    )


if (
    video_predictions[
        "sample_count"
    ].max() == 1
):
    print(
        "\nIMPORTANT: Each video_id contains only one sample. "
        "Therefore, this metadata does not provide genuine "
        "multi-frame video aggregation. These results must "
        "be described as video-ID-grouped auxiliary results, "
        "not as independent video-level performance."
    )

In [ ]:
# ============================================================
# 22. FINAL QUALITY GATES
# ============================================================

import torch


def validate_completed_experiment():
    required_output_files = [
        MODE_DIRS["checkpoints"]
        / "last.ckpt",

        MODE_DIRS["checkpoints"]
        / "best.ckpt",

        MODE_DIRS["metrics"]
        / "epoch_metrics.csv",

        MODE_DIRS["metrics"]
        / "test_metrics.json",

        MODE_DIRS["metrics"]
        / "video_level_metrics.json",

        MODE_DIRS["predictions"]
        / "test_predictions.csv",

        MODE_DIRS["predictions"]
        / "video_level_predictions.csv",
    ]


    missing_output_files = [
        path
        for path in required_output_files
        if not path.exists()
    ]


    if missing_output_files:
        raise AssertionError(
            "Required experiment outputs are missing:\n"
            + "\n".join(
                str(path)
                for path in missing_output_files
            )
        )


    # --------------------------------------------------------
    # Checkpoint integrity
    # --------------------------------------------------------

    best_checkpoint = torch.load(
        MODE_DIRS["checkpoints"]
        / "best.ckpt",
        map_location="cpu",
        weights_only=False,
    )


    required_checkpoint_keys = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "best_metric_score",
        "best_threshold",
        "config",
        "history",
        "torch_rng_state",
    }


    missing_checkpoint_keys = (
        required_checkpoint_keys
        - set(best_checkpoint.keys())
    )


    if missing_checkpoint_keys:
        raise AssertionError(
            "Checkpoint keys are missing: "
            f"{sorted(missing_checkpoint_keys)}"
        )


    # --------------------------------------------------------
    # Epoch metric integrity
    # --------------------------------------------------------

    epoch_metrics = pd.read_csv(
        MODE_DIRS["metrics"]
        / "epoch_metrics.csv"
    )


    numeric_columns = (
        epoch_metrics
        .select_dtypes(
            include=[np.number]
        )
        .columns
    )


    if not np.isfinite(
        epoch_metrics[
            numeric_columns
        ].to_numpy()
    ).all():
        raise FloatingPointError(
            "NaN or Inf detected in epoch metrics."
        )


    # --------------------------------------------------------
    # Test prediction integrity
    # --------------------------------------------------------

    test_predictions = pd.read_csv(
        MODE_DIRS["predictions"]
        / "test_predictions.csv"
    )


    assert len(test_predictions) == 302, (
        "Unexpected test sample count."
    )


    assert (
        test_predictions["sample_id"]
        .is_unique
    ), "Duplicated test sample IDs detected."


    assert test_predictions[
        "probability_fake"
    ].between(
        0,
        1,
        inclusive="both",
    ).all(), (
        "Invalid test probability detected."
    )


    # --------------------------------------------------------
    # Test metric integrity
    # --------------------------------------------------------

    with open(
        MODE_DIRS["metrics"]
        / "test_metrics.json",
        "r",
        encoding="utf-8",
    ) as file:
        test_metrics = json.load(file)


    expected_metric_names = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
    ]


    for metric_name in expected_metric_names:
        metric_value = float(
            test_metrics[
                metric_name
            ]
        )

        assert 0.0 <= metric_value <= 1.0, (
            f"Invalid metric value: "
            f"{metric_name}={metric_value}"
        )


    # --------------------------------------------------------
    # Figure quality
    # --------------------------------------------------------

    figure_files = sorted(
        MODE_DIRS["figures"]
        .glob("*.png")
    )


    assert len(figure_files) == 6, (
        f"Expected 6 figures, "
        f"found {len(figure_files)}."
    )


    figure_quality = {}


    for figure_path in figure_files:
        with Image.open(
            figure_path
        ) as image:
            image_size = image.size

        if min(image_size) < 600:
            raise AssertionError(
                f"Figure resolution is insufficient: "
                f"{figure_path.name} -> {image_size}"
            )

        figure_quality[
            figure_path.name
        ] = {
            "resolution":
                list(image_size),

            "minimum_short_edge_600":
                "PASSED",
        }


    quality_results = {
        "roi_type":
            "mouth",

        "mode":
            "full",

        "required_outputs":
            "PASSED",

        "checkpoint_integrity":
            "PASSED",

        "numeric_metrics":
            "PASSED",

        "test_prediction_count":
            int(
                len(test_predictions)
            ),

        "prediction_uniqueness":
            "PASSED",

        "probability_range":
            "PASSED",

        "figure_count":
            int(
                len(figure_files)
            ),

        "figure_quality":
            figure_quality,

        "best_epoch":
            int(
                test_metrics[
                    "best_epoch"
                ]
            ),

        "selected_threshold":
            float(
                test_metrics[
                    "selected_threshold"
                ]
            ),

        "final_status":
            "EXPERIMENT COMPLETED SUCCESSFULLY",
    }


    audit_directory = (
        RUN_DIR / "audit"
    )

    audit_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    atomic_text_write(
        json.dumps(
            quality_results,
            indent=2,
            ensure_ascii=False,
        ),
        (
            audit_directory
            / "final_quality_gates.json"
        ),
    )


    return quality_results


final_quality_results = (
    validate_completed_experiment()
)


print(
    json.dumps(
        final_quality_results,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nExperiment completed successfully.")
print("Final RUN directory:")
print(RUN_DIR)